# Batch Star Formation Rate (SFR) Map Calculation with HEALPix

This notebook contains the complete, reproducible pipeline for calculating and exporting integrated galaxy Star Formation Rate (SFR) maps on the celestial sphere. It filters the local universe galaxy catalog across multiple distance thresholds ($30 \text{ to } 200 \text{ Mpc}$) and projects the integrated SFR onto a spherical HEALPix grid over various angular resolutions ($\text{NSIDE} = 1 \text{ to } 256$).

## Computational Environment & Dependencies

To ensure absolute scientific reproducibility, this notebook requires specific astronomical coordinate transformation libraries alongside standard data science packages. The exact versions used during development are pinned in the root directory's `requirements.txt`.

### Core Astronomical & Numerical Packages:
* **Python Version:** `3.10` or higher (recommended)
* **healpy (v1.16.x or higher):** Python wrapper for HEALPix (Hierarchical Equal Area isoLatitude Pixelation). Used for fast spherical pixel querying (`ang2pix`), pixel coordinate resolution (`pix2ang`), and map generation (`nside2npix`).
* **pandas & numpy:** Used for high-efficiency memory handling of the full galaxy catalog and fast array-based vectorization (`np.bincount`) during SFR summation.
* **matplotlib:** (Optional dependency via healpy) Used internally for coordinate projections and visual validation.

### Core Computational Logic Note:
To avoid physical bias during pixel accumulation, the script converts logarithmic star formation rates ($\text{final\_logSFR}$) into linear space ($10^{\text{final\_logSFR}}$), performs a fast histogram summation via `np.bincount` inside each equal-area HEALPix pixel, and seamlessly transforms the total integrated pixel SFR back into standard logarithmic form ($\log \text{SFR}$).

### Environment Reproduction:
You can quickly set up your local or server environment to match this notebook using:
```bash
pip install -r requirements.txt

In [1]:
"""
Batch SFR Map Calculation with HEALPix
Workflow:
    1. Load galaxy catalog (read once for efficiency)
    2. Iterate over multiple HEALPix resolutions (NSIDE)
    3. Iterate over distance thresholds (30 ~ 200 Mpc, step = 10)
    4. Compute total SFR per HEALPix pixel
    5. Save results to TXT and NPZ files automatically

HEALPix Angular Resolution Reference:
    NSIDE -> Angular pixel size
"""
import os
import warnings
import numpy as np
import pandas as pd
import healpy as hp
import matplotlib.pyplot as plt

# -------------------------- Global Configuration & Constants --------------------------
# Suppress non-critical warnings
warnings.filterwarnings("ignore")

# NSIDE to angular pixel size mapping
NSIDE_THETA_MAPPING = {
    1: '58.6deg',
    2: '29.3deg',
    4: '14.7deg',
    8: '7.33deg',
    16: '3.66deg',
    32: '1.83deg',
    64: '55arcmin',
    128: '27.5arcmin'
    # 256: '13.7arcmin',
    # 512: '6.87arcmin',
    # 1024: '3.44arcmin',
    # 2048: '1.72arcmin'
}

# Select target NSIDE list for batch processing
TARGET_NSIDES = [1, 2, 4, 8, 16, 32, 64, 128]

# Distance range: 30 ~ 200 Mpc, step = 10 Mpc
DISTANCE_RANGE = range(30, 210, 10)

# Fixed computational constants
DEFAULT_FILL_VALUE = 1e-10    # Fill value for empty HEALPix pixels
ROUND_DECIMALS = 2             # Decimal places for final logSFR

# File paths (modify here according to your environment)
INPUT_CATALOG_PATH = "/Users/cyh/Mass_SFR/newVersion/data/Allgalaxy_final_version.csv"
BASE_OUTPUT_DIR = "/Users/cyh/Mass_SFR/newVersion/data/mapdate/SFR"

# -------------------------- Core Function Definition --------------------------
def compute_and_export_sfr_map(
    nside: int,
    theta_label: str,
    distance_cut: float,
    raw_data: pd.DataFrame
) -> None:
    """
    Calculate integrated SFR for each HEALPix pixel and export results to files.

    Parameters
    ----------
    nside : int
        HEALPix NSIDE resolution parameter
    theta_label : str
        Angular resolution label for file naming & directory
    distance_cut : float
        Maximum distance threshold (Mpc), only galaxies with D <= cut are used
    raw_data : pd.DataFrame
        Full loaded galaxy catalog dataframe

    Outputs
    -------
    TXT file & NPZ file containing: pixel ID, RA, Dec, log10(total SFR)
    Files are saved to: {BASE_OUTPUT_DIR}/{theta_label}/
    """
    # Step 1: Filter galaxies by distance
    dist_mask = raw_data["dL/Mpc"] <= distance_cut
    filtered_df = raw_data[dist_mask]

    ra_arr = filtered_df["ra"].values
    dec_arr = filtered_df["dec"].values

    # Convert logSFR to linear SFR
    log_sfr = filtered_df["final_logSFR"].values
    sfr_linear = 10 ** log_sfr

    # Step 2: HEALPix pixel initialization
    npix = hp.nside2npix(nside)
    sfr_map = np.full(npix, DEFAULT_FILL_VALUE, dtype=np.float64)

    # Get pixel index for each galaxy (lonlat=True: RA, Dec in degrees)
    pixel_idx = hp.ang2pix(nside, ra_arr, dec_arr, lonlat=True)

    # Sum linear SFR within each pixel (high-performance bincount)
    sfr_sum_per_pixel = np.bincount(pixel_idx, weights=sfr_linear, minlength=npix)

    # Convert total linear SFR back to log10(SFR), keep 2 decimal places
    valid_pixel_mask = sfr_sum_per_pixel > 0
    sfr_map[valid_pixel_mask] = np.round(
        np.log10(sfr_sum_per_pixel[valid_pixel_mask]),
        decimals=ROUND_DECIMALS
    )

    # Step 3: Calculate coordinates for all pixel centers
    all_pixels = np.arange(npix)
    ra_center, dec_center = hp.pix2ang(nside, all_pixels, lonlat=True)

    # Step 4: Create output directory (auto-create if not exists)
    output_subdir = os.path.join(BASE_OUTPUT_DIR, theta_label)
    os.makedirs(output_subdir, exist_ok=True)

    # Define file paths
    # txt_filename = f"SFR_D{distance_cut}_{theta_label}.txt"
    npz_filename = f"SFR_D{distance_cut}_{theta_label}.npz"
    # txt_full_path = os.path.join(output_subdir, txt_filename)
    npz_full_path = os.path.join(output_subdir, npz_filename)

    # # Save TXT file: [pixel_id, RA, Dec, logSFR]
    # output_txt_data = np.column_stack((all_pixels, ra_center, dec_center, sfr_map))
    # np.savetxt(txt_full_path, output_txt_data)

    # Save NPZ file (structured data for fast loading)
    np.savez(
        npz_full_path,
        npix=all_pixels,
        ra=ra_center,
        dec=dec_center,
        value=sfr_map
    )

    # Print progress log
    print(
        f"Completed | NSIDE={nside:3d} | Res={theta_label:12s} | Dist={distance_cut:3d} Mpc "
        f"| Galaxies={len(ra_arr):6d}"
    )

# -------------------------- Main Workflow --------------------------
def main():
    print("=" * 60)
    print("Start Batch HEALPix SFR Map Calculation")
    print("=" * 60)

    # Check if input file exists
    if not os.path.isfile(INPUT_CATALOG_PATH):
        raise FileNotFoundError(f"Input catalog not found: {INPUT_CATALOG_PATH}")

    # Load galaxy catalog (read only once to optimize I/O)
    print(f"Loading galaxy catalog from: {INPUT_CATALOG_PATH}")
    galaxy_data = pd.read_csv(INPUT_CATALOG_PATH, low_memory=False)
    total_galaxies = len(galaxy_data)
    print(f"Total galaxies in raw catalog: {total_galaxies}\n")

    # Batch loop over all target NSIDE resolutions
    for nside in TARGET_NSIDES:
        # Get corresponding angular resolution label
        res_label = NSIDE_THETA_MAPPING[nside]
        print(f"----- Processing NSIDE = {nside} (Angular Resolution: {res_label}) -----")

        # Loop over all distance thresholds
        for dist in DISTANCE_RANGE:
            compute_and_export_sfr_map(
                nside=nside,
                theta_label=res_label,
                distance_cut=dist,
                raw_data=galaxy_data
            )
        print("")

    # Final log
    print("=" * 60)
    print("All batch tasks finished successfully!")
    print(f"All results saved to: {BASE_OUTPUT_DIR}")
    print("=" * 60)

# Standard Python entry point (for module import & standalone run)
if __name__ == "__main__":
    main()

Start Batch HEALPix SFR Map Calculation
Loading galaxy catalog from: /Users/cyh/Mass_SFR/newVersion/data/Allgalaxy_final_version.csv
Total galaxies in raw catalog: 364148

----- Processing NSIDE = 1 (Angular Resolution: 58.6deg) -----
Completed | NSIDE=  1 | Res=58.6deg      | Dist= 30 Mpc | Galaxies=  7789
Completed | NSIDE=  1 | Res=58.6deg      | Dist= 40 Mpc | Galaxies= 11424
Completed | NSIDE=  1 | Res=58.6deg      | Dist= 50 Mpc | Galaxies= 15798
Completed | NSIDE=  1 | Res=58.6deg      | Dist= 60 Mpc | Galaxies= 21791
Completed | NSIDE=  1 | Res=58.6deg      | Dist= 70 Mpc | Galaxies= 29983
Completed | NSIDE=  1 | Res=58.6deg      | Dist= 80 Mpc | Galaxies= 39687
Completed | NSIDE=  1 | Res=58.6deg      | Dist= 90 Mpc | Galaxies= 50268
Completed | NSIDE=  1 | Res=58.6deg      | Dist=100 Mpc | Galaxies= 64427
Completed | NSIDE=  1 | Res=58.6deg      | Dist=110 Mpc | Galaxies= 81488
Completed | NSIDE=  1 | Res=58.6deg      | Dist=120 Mpc | Galaxies= 99502
Completed | NSIDE=  1 | R